# 🅿️ ParkWise: ML-Driven Dynamic Pricing for Urban Parking

**Capstone Project — Summer Analytics 2025**  
*Consulting & Analytics Club × Pathway*

---

## 📋 Table of Contents

| # | Section | DS Skill |
|---|---------|----------|
| 1 | [Setup & Data Loading](#1) | Data Engineering |
| 2 | [Deep EDA & Statistical Analysis](#2) | Statistical Thinking |
| 3 | [Time-Series Demand Forecasting](#3) | Time-Series ML |
| 4 | [ML-Based Pricing Model](#4) | Supervised Learning |
| 5 | [Demand Segmentation (Clustering)](#5) | Unsupervised Learning |
| 6 | [Geospatial Competitor Analysis](#6) | Geospatial Analysis |
| 7 | [Price Elasticity Estimation](#7) | Econometrics |
| 8 | [A/B Testing & Simulation](#8) | Experiment Design |
| 9 | [Feature Importance & Explainability](#9) | Model Interpretability |
| 10 | [Conclusions & Key Findings](#10) | Business Insights |

---

## 🎯 Problem Statement

Design an intelligent **real-time dynamic pricing system** for urban parking lots that adjusts prices based on:
- **Demand signals**: Occupancy rate, queue length, traffic conditions
- **Temporal patterns**: Hour of day, day of week, special events
- **Spatial competition**: Proximity and pricing of competitor lots
- **Vehicle segmentation**: Different pricing for cars, bikes, trucks, cycles

**Objective**: Maximize revenue while maintaining fair utilization and price stability.

**Dataset**: 14 parking lots × 73 days × 18 timestamps/day = **18,368 records** with occupancy, queue, traffic, vehicle type, location (lat/long), and event data.

---
<a id="1"></a>
## 1️⃣ Setup & Data Loading

### 1.1 Install Dependencies

In [ ]:
# Install required packages (run once)
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost statsmodels shap folium scipy --quiet

### 1.2 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

# Set professional plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 100,
})

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("✅ All libraries imported successfully!")

### 1.3 Load & Preview Dataset

In [ ]:
# Load the parking dataset
df = pd.read_csv('dataset.csv')

print(f"📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"📍 Unique Parking Lots: {df['SystemCodeNumber'].nunique()}")
print(f"📅 Date Range: {df['LastUpdatedDate'].min()} to {df['LastUpdatedDate'].max()}")
print(f"🚗 Vehicle Types: {df['VehicleType'].unique().tolist()}")
print(f"🚦 Traffic Levels: {df['TrafficConditionNearby'].unique().tolist()}")
print()
df.head(10)

In [ ]:
# Dataset info and statistics
print("="*60)
print("📋 DATASET INFO")
print("="*60)
print(f"\nData Types:\n{df.dtypes}")
print(f"\nNull Values:\n{df.isnull().sum()}")
print(f"\nBasic Statistics:")
df.describe()

### 1.4 Feature Engineering Pipeline

We engineer several categories of features:
1. **Temporal**: Hour, day of week, cyclical sin/cos encodings
2. **Demand**: Occupancy rate, lag features, rolling statistics  
3. **Contextual**: Traffic (numeric), vehicle weight, special day flag
4. **Synthetic price**: Multi-factor pricing formula for ML training target

In [ ]:
# ── Core Feature Engineering ──

# Map categorical features to numeric
traffic_map = {'low': 1, 'average': 2, 'high': 3}
vehicle_weights = {'car': 1.0, 'bike': 0.5, 'cycle': 0.3, 'truck': 1.5}

df['Traffic'] = df['TrafficConditionNearby'].map(traffic_map)
df['VehicleWeight'] = df['VehicleType'].map(vehicle_weights)

# Parse timestamp
df['Timestamp'] = pd.to_datetime(
    df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime'],
    format='%d-%m-%Y %H:%M:%S'
)
df = df.sort_values('Timestamp').reset_index(drop=True)

# Compute occupancy rate (core demand metric)
df['OccupancyRate'] = df['Occupancy'] / df['Capacity'].replace(0, 1)
df['OccupancyRate'] = df['OccupancyRate'].clip(0, 1)

# ── Temporal Features ──
df['Hour'] = df['Timestamp'].dt.hour
df['DayOfWeek'] = df['Timestamp'].dt.dayofweek
df['Month'] = df['Timestamp'].dt.month
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
df['DayName'] = df['Timestamp'].dt.day_name()

# Cyclical encoding (preserves circular nature of time)
df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
df['DayOfWeek_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['DayOfWeek_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)

# ── Lag Features (per parking lot) ──
for lag in [1, 2, 3, 6]:
    df[f'OccupancyRate_lag_{lag}'] = df.groupby('SystemCodeNumber')['OccupancyRate'].shift(lag)

# ── Rolling Features (per parking lot) ──
for window in [3, 6, 12]:
    df[f'OccupancyRate_rolling_mean_{window}'] = df.groupby('SystemCodeNumber')['OccupancyRate'].transform(
        lambda x: x.rolling(window, min_periods=1).mean()
    )
    df[f'OccupancyRate_rolling_std_{window}'] = df.groupby('SystemCodeNumber')['OccupancyRate'].transform(
        lambda x: x.rolling(window, min_periods=1).std()
    ).fillna(0)

# ── Synthetic Price Target (for ML training) ──
np.random.seed(42)
occupancy_factor = 1.0 + 2.5 * (df['OccupancyRate'] ** 1.5)
queue_factor = 1.0 + 0.15 * df['QueueLength']
traffic_factor = 1.0 + 0.2 * (df['Traffic'] - 1)
special_day_factor = 1.0 + 0.5 * df['IsSpecialDay']
vehicle_factor = df['VehicleWeight']
peak_morning = np.exp(-0.5 * ((df['Hour'] - 9) / 2) ** 2) * 0.3
peak_evening = np.exp(-0.5 * ((df['Hour'] - 18) / 2) ** 2) * 0.3
time_surge = 1.0 + peak_morning + peak_evening
weekend_factor = 1.0 + 0.15 * df['IsWeekend']

df['Price'] = (
    10.0 * occupancy_factor * queue_factor * traffic_factor
    * special_day_factor * vehicle_factor * time_surge * weekend_factor
    + np.random.normal(0, 0.5, len(df))
).clip(2.0, 100.0).round(2)

# ── Demand Score (composite metric for clustering) ──
df['DemandScore'] = (
    0.50 * df['OccupancyRate']
    + 0.25 * (df['QueueLength'] / df['QueueLength'].max())
    + 0.15 * ((df['Traffic'] - 1) / 2)
    + 0.10 * df['IsSpecialDay']
) * 100

print(f"✅ Feature engineering complete!")
print(f"   Total features: {len(df.columns)}")
print(f"   New columns: {[c for c in df.columns if c not in ['ID','SystemCodeNumber','Capacity','Latitude','Longitude','Occupancy','VehicleType','TrafficConditionNearby','QueueLength','IsSpecialDay','LastUpdatedDate','LastUpdatedTime']]}")
print(f"\n📊 Engineered DataFrame shape: {df.shape}")
df.head()

---
<a id="2"></a>
## 2️⃣ Deep EDA & Statistical Analysis

> **DS Skill Demonstrated**: Statistical thinking, hypothesis testing, data storytelling

Before building any models, we must deeply understand the data. This section covers:
- Distribution analysis of key features
- Correlation analysis to identify demand drivers
- Temporal pattern decomposition
- Statistical hypothesis testing

### 2.1 Feature Distributions

In [ ]:
# Distribution of key numerical features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

features = {
    'OccupancyRate': 'Occupancy Rate',
    'QueueLength': 'Queue Length',
    'Traffic': 'Traffic Level',
    'VehicleWeight': 'Vehicle Weight',
    'Price': 'Synthetic Price ($)',
    'DemandScore': 'Demand Score',
}

for ax, (col, title) in zip(axes.ravel(), features.items()):
    ax.hist(df[col], bins=40, alpha=0.7, color='#4C72B0', edgecolor='white', density=True)
    df[col].plot.kde(ax=ax, color='#C44E52', linewidth=2)
    ax.axvline(df[col].mean(), color='#DD8452', linestyle='--', linewidth=2,
               label=f'Mean: {df[col].mean():.2f}')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Distribution of Key Features', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

### 2.2 Occupancy Distribution by Parking Lot

In [ ]:
# Box plot: Occupancy rate per parking lot
fig, ax = plt.subplots(figsize=(16, 6))

lot_data = [group['OccupancyRate'].values for _, group in df.groupby('SystemCodeNumber')]
lot_names = [name for name, _ in df.groupby('SystemCodeNumber')]

bp = ax.boxplot(lot_data, vert=True, patch_artist=True, widths=0.6)
colors = plt.cm.Set3(np.linspace(0, 1, len(lot_names)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.set_xticklabels([n[:10] for n in lot_names], rotation=45, ha='right')
ax.set_title('Occupancy Rate Distribution by Parking Lot', fontweight='bold', fontsize=14)
ax.set_ylabel('Occupancy Rate')
ax.axhline(df['OccupancyRate'].mean(), color='red', linestyle='--', alpha=0.7,
           label=f'Overall Mean: {df["OccupancyRate"].mean():.3f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.3 Correlation Heatmap

> Which features are most correlated with demand and pricing?

In [ ]:
# Correlation heatmap
numeric_cols = ['OccupancyRate', 'QueueLength', 'Traffic', 'IsSpecialDay',
                'VehicleWeight', 'Capacity', 'Hour', 'DayOfWeek', 'IsWeekend', 'Price']

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(250, 10, as_cmap=True)

sns.heatmap(
    corr_matrix, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
    annot=True, fmt='.2f', square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8}, ax=ax,
)
ax.set_title('Feature Correlation Heatmap', fontweight='bold', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# Top correlations with Price
print("\n📊 Top Feature Correlations with Price:")
price_corr = corr_matrix['Price'].drop('Price').abs().sort_values(ascending=False)
for feat, corr_val in price_corr.items():
    print(f"   {feat:25s} → {corr_val:.4f}")

### 2.4 Temporal Patterns

In [ ]:
# Hourly and daily patterns
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Average occupancy by hour
hourly = df.groupby('Hour')['OccupancyRate'].agg(['mean', 'std']).reset_index()
axes[0,0].fill_between(hourly['Hour'], hourly['mean'] - hourly['std'],
                        hourly['mean'] + hourly['std'], alpha=0.2, color='#4C72B0')
axes[0,0].plot(hourly['Hour'], hourly['mean'], 'o-', color='#4C72B0', linewidth=2, markersize=6)
axes[0,0].set_title('Average Occupancy Rate by Hour', fontweight='bold')
axes[0,0].set_xlabel('Hour of Day')
axes[0,0].set_ylabel('Occupancy Rate')
axes[0,0].set_xticks(range(0, 24))

# 2. Heatmap: Hour × Day of Week
pivot = df.pivot_table(values='OccupancyRate', index='DayOfWeek', columns='Hour', aggfunc='mean')
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
sns.heatmap(pivot, cmap='YlOrRd', annot=False, ax=axes[0,1],
            yticklabels=day_labels[:len(pivot)], cbar_kws={'label': 'Avg Occupancy Rate'})
axes[0,1].set_title('Occupancy: Hour × Day of Week', fontweight='bold')

# 3. Average by day of week
daily = df.groupby('DayOfWeek')['OccupancyRate'].mean()
bar_colors = ['#4C72B0'] * 5 + ['#C44E52'] * 2
axes[1,0].bar(range(len(daily)), daily.values, color=bar_colors[:len(daily)], edgecolor='white')
axes[1,0].set_xticks(range(len(daily)))
axes[1,0].set_xticklabels(day_labels[:len(daily)])
axes[1,0].set_title('Average Occupancy by Day of Week', fontweight='bold')
axes[1,0].set_ylabel('Occupancy Rate')

# 4. Daily time series trend
daily_ts = df.groupby(df['Timestamp'].dt.date)['OccupancyRate'].mean()
axes[1,1].plot(daily_ts.index, daily_ts.values, color='#4C72B0', linewidth=1.5)
axes[1,1].fill_between(daily_ts.index, daily_ts.values, alpha=0.3, color='#4C72B0')
axes[1,1].set_title('Daily Average Occupancy Over Time', fontweight='bold')
axes[1,1].set_xlabel('Date')
axes[1,1].set_ylabel('Average Occupancy Rate')
axes[1,1].tick_params(axis='x', rotation=45)

plt.suptitle('Temporal Demand Patterns', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

### 2.5 Parking Lot Comparison

In [ ]:
# Compare all 14 parking lots across key metrics
lot_stats = df.groupby('SystemCodeNumber').agg(
    avg_occupancy=('OccupancyRate', 'mean'),
    max_occupancy=('OccupancyRate', 'max'),
    avg_queue=('QueueLength', 'mean'),
    capacity=('Capacity', 'first'),
    special_day_pct=('IsSpecialDay', 'mean'),
).reset_index().sort_values('avg_occupancy', ascending=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Avg Occupancy
colors_occ = plt.cm.RdYlGn_r(lot_stats['avg_occupancy'])
axes[0,0].barh(range(len(lot_stats)), lot_stats['avg_occupancy'], color=colors_occ, edgecolor='white')
axes[0,0].set_yticks(range(len(lot_stats)))
axes[0,0].set_yticklabels([n[:12] for n in lot_stats['SystemCodeNumber']])
axes[0,0].set_title('Avg Occupancy Rate by Lot', fontweight='bold')
axes[0,0].set_xlabel('Occupancy Rate')

# 2. Avg Queue
axes[0,1].barh(range(len(lot_stats)), lot_stats['avg_queue'],
               color=plt.cm.Blues(np.linspace(0.3, 0.9, len(lot_stats))), edgecolor='white')
axes[0,1].set_yticks(range(len(lot_stats)))
axes[0,1].set_yticklabels([n[:12] for n in lot_stats['SystemCodeNumber']])
axes[0,1].set_title('Avg Queue Length by Lot', fontweight='bold')

# 3. Capacity
axes[1,0].barh(range(len(lot_stats)), lot_stats['capacity'],
               color=plt.cm.Oranges(np.linspace(0.3, 0.9, len(lot_stats))), edgecolor='white')
axes[1,0].set_yticks(range(len(lot_stats)))
axes[1,0].set_yticklabels([n[:12] for n in lot_stats['SystemCodeNumber']])
axes[1,0].set_title('Capacity by Lot', fontweight='bold')

# 4. Vehicle type distribution
vtype_counts = df.groupby('VehicleType').size()
axes[1,1].pie(vtype_counts.values, labels=vtype_counts.index, autopct='%1.1f%%',
              colors=plt.cm.Set2.colors[:len(vtype_counts)], startangle=90)
axes[1,1].set_title('Vehicle Type Distribution', fontweight='bold')

plt.suptitle('Parking Lot Comparison Dashboard', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

### 2.6 Statistical Hypothesis Testing

We test four key hypotheses to validate assumptions about the data:

1. **H₀**: Mean occupancy is the same on special vs normal days (Welch's t-test)
2. **H₀**: Occupancy distributions are identical for special vs normal days (Mann-Whitney U)
3. **H₀**: Mean occupancy is the same across all parking lots (One-way ANOVA)
4. **H₀**: Traffic condition is independent of special day status (Chi-squared)

In [ ]:
# ══════════════════════════════════════════════════════════════
# STATISTICAL HYPOTHESIS TESTS
# ══════════════════════════════════════════════════════════════

special = df[df['IsSpecialDay'] == 1]['OccupancyRate']
normal = df[df['IsSpecialDay'] == 0]['OccupancyRate']

# Test 1: Welch's t-test (Special Day vs Normal Day)
t_stat, t_pval = stats.ttest_ind(special, normal, equal_var=False)
print("=" * 60)
print("📊 Test 1: Welch's t-test (Special Day Occupancy)")
print("=" * 60)
print(f"   H₀: Mean occupancy is the same on special vs normal days")
print(f"   Special Day Mean: {special.mean():.4f}")
print(f"   Normal Day Mean:  {normal.mean():.4f}")
print(f"   t-statistic: {t_stat:.4f}")
print(f"   p-value: {t_pval:.6f}")
print(f"   Verdict (α=0.05): {'✅ REJECT H₀ (significant)' if t_pval < 0.05 else '❌ FAIL TO REJECT H₀'}")

# Test 2: Mann-Whitney U (non-parametric)
u_stat, u_pval = stats.mannwhitneyu(special, normal, alternative='two-sided')
print(f"\n{'='*60}")
print("📊 Test 2: Mann-Whitney U (Non-parametric)")
print("=" * 60)
print(f"   U-statistic: {u_stat:.4f}")
print(f"   p-value: {u_pval:.6f}")
print(f"   Verdict: {'✅ REJECT H₀' if u_pval < 0.05 else '❌ FAIL TO REJECT H₀'}")

# Test 3: One-way ANOVA (across lots)
lot_groups = [group['OccupancyRate'].values for _, group in df.groupby('SystemCodeNumber')]
f_stat, f_pval = stats.f_oneway(*lot_groups)
print(f"\n{'='*60}")
print("📊 Test 3: One-way ANOVA (Occupancy Across Lots)")
print("=" * 60)
print(f"   H₀: Mean occupancy is the same across all parking lots")
print(f"   F-statistic: {f_stat:.4f}")
print(f"   p-value: {f_pval:.6f}")
print(f"   Verdict: {'✅ REJECT H₀ (significant)' if f_pval < 0.05 else '❌ FAIL TO REJECT H₀'}")

# Test 4: Chi-squared (Traffic vs Special Day)
contingency = pd.crosstab(df['TrafficConditionNearby'], df['IsSpecialDay'])
chi2, chi_pval, dof, expected = stats.chi2_contingency(contingency)
print(f"\n{'='*60}")
print("📊 Test 4: Chi-squared (Traffic × Special Day)")
print("=" * 60)
print(f"   H₀: Traffic condition is independent of special day status")
print(f"   χ²-statistic: {chi2:.4f}")
print(f"   p-value: {chi_pval:.6f}")
print(f"   Degrees of Freedom: {dof}")
print(f"   Verdict: {'✅ REJECT H₀ (significant)' if chi_pval < 0.05 else '❌ FAIL TO REJECT H₀'}")

# Summary table
print(f"\n\n{'='*60}")
print("📋 HYPOTHESIS TESTING SUMMARY")
print("=" * 60)
test_summary = pd.DataFrame({
    'Test': ["Welch's t-test", "Mann-Whitney U", "One-way ANOVA", "Chi-squared"],
    'Hypothesis': [
        "Special day occupancy = Normal day",
        "Same distribution (special vs normal)",
        "Same occupancy across all lots",
        "Traffic independent of special day",
    ],
    'Statistic': [f"{t_stat:.4f}", f"{u_stat:.4f}", f"{f_stat:.4f}", f"{chi2:.4f}"],
    'P-value': [f"{t_pval:.6f}", f"{u_pval:.6f}", f"{f_pval:.6f}", f"{chi_pval:.6f}"],
    'Significant': [t_pval < 0.05, u_pval < 0.05, f_pval < 0.05, chi_pval < 0.05],
})
test_summary

---
<a id="3"></a>
## 3️⃣ Time-Series Demand Forecasting

> **DS Skill Demonstrated**: Time-series modeling, feature engineering (lag features, rolling means, cyclical encoding)

**Goal**: Predict future occupancy/demand for each parking lot.

**Models compared**:
- **ARIMA(2,1,2)** — Classical statistical time-series model
- **SARIMA** — Seasonal ARIMA with daily seasonality
- **XGBoost with Lag Features** — Gradient boosting with temporal features

**Evaluation metrics**: RMSE, MAE, MAPE

### 3.1 Prepare Time Series Data

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

# Select a representative parking lot for detailed analysis
target_lot = df['SystemCodeNumber'].value_counts().index[0]  # Most data points
print(f"🎯 Target lot for forecasting: {target_lot}")

# Extract hourly time series for the target lot
lot_df = df[df['SystemCodeNumber'] == target_lot].copy()
lot_df = lot_df.set_index('Timestamp')
series = lot_df['OccupancyRate'].resample('h').mean().ffill().bfill()

print(f"   Series length: {len(series)} hourly observations")
print(f"   Date range: {series.index.min()} → {series.index.max()}")
print(f"   Mean occupancy: {series.mean():.4f}")

# Plot the time series
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(series.index, series.values, color='#4C72B0', linewidth=0.8, alpha=0.8)
ax.fill_between(series.index, series.values, alpha=0.2, color='#4C72B0')
ax.set_title(f'Hourly Occupancy Rate — {target_lot}', fontweight='bold')
ax.set_xlabel('Time')
ax.set_ylabel('Occupancy Rate')
plt.tight_layout()
plt.show()

### 3.2 ARIMA Model

In [ ]:
# ── ARIMA(2,1,2) ──
test_size = 48  # Last 48 hours as test set
train_series = series[:-test_size]
test_series = series[-test_size:]

print("📈 Training ARIMA(2,1,2)...")
arima_model = ARIMA(train_series, order=(2, 1, 2))
arima_fitted = arima_model.fit()
arima_pred = arima_fitted.forecast(steps=test_size)

# Metrics
arima_rmse = np.sqrt(mean_squared_error(test_series, arima_pred))
arima_mae = mean_absolute_error(test_series, arima_pred)
mask = test_series != 0
arima_mape = np.mean(np.abs((test_series[mask] - arima_pred[mask]) / test_series[mask])) * 100

print(f"   ✅ ARIMA Results:")
print(f"      RMSE:  {arima_rmse:.4f}")
print(f"      MAE:   {arima_mae:.4f}")
print(f"      MAPE:  {arima_mape:.2f}%")

### 3.3 SARIMA Model (Seasonal)

In [ ]:
# ── SARIMA with daily seasonality ──
seasonal_period = min(24, len(train_series) // 4)
if seasonal_period < 2:
    seasonal_period = 2

print(f"📈 Training SARIMA(1,1,1)(1,0,1,{seasonal_period})...")
sarima_model = SARIMAX(
    train_series, order=(1, 1, 1),
    seasonal_order=(1, 0, 1, seasonal_period),
    enforce_stationarity=False, enforce_invertibility=False
)
sarima_fitted = sarima_model.fit(disp=False, maxiter=200)
sarima_pred = sarima_fitted.forecast(steps=test_size)

sarima_rmse = np.sqrt(mean_squared_error(test_series, sarima_pred))
sarima_mae = mean_absolute_error(test_series, sarima_pred)
sarima_mape = np.mean(np.abs((test_series[mask] - sarima_pred[mask]) / test_series[mask])) * 100

print(f"   ✅ SARIMA Results:")
print(f"      RMSE:  {sarima_rmse:.4f}")
print(f"      MAE:   {sarima_mae:.4f}")
print(f"      MAPE:  {sarima_mape:.2f}%")

### 3.4 XGBoost with Lag Features

In [ ]:
# ── XGBoost Forecaster ──
# Create feature matrix with lag and rolling features
def create_ts_features(s, lags=[1,2,3,6,12,24], windows=[3,6,12,24]):
    feat = pd.DataFrame(index=s.index)
    feat['target'] = s.values
    for lag in lags:
        feat[f'lag_{lag}'] = s.shift(lag)
    for w in windows:
        feat[f'rolling_mean_{w}'] = s.rolling(w, min_periods=1).mean()
        feat[f'rolling_std_{w}'] = s.rolling(w, min_periods=1).std().fillna(0)
    feat['hour'] = s.index.hour
    feat['dow'] = s.index.dayofweek
    feat['hour_sin'] = np.sin(2 * np.pi * feat['hour'] / 24)
    feat['hour_cos'] = np.cos(2 * np.pi * feat['hour'] / 24)
    feat['dow_sin'] = np.sin(2 * np.pi * feat['dow'] / 7)
    feat['dow_cos'] = np.cos(2 * np.pi * feat['dow'] / 7)
    return feat.dropna()

feat_df = create_ts_features(series)
X_ts = feat_df.drop(columns=['target'])
y_ts = feat_df['target']

X_train_ts = X_ts[:-test_size]
X_test_ts = X_ts[-test_size:]
y_train_ts = y_ts[:-test_size]
y_test_ts = y_ts[-test_size:]

print("📈 Training XGBoost with lag features...")
xgb_ts = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_ts.fit(X_train_ts, y_train_ts, eval_set=[(X_test_ts, y_test_ts)], verbose=False)
xgb_pred = xgb_ts.predict(X_test_ts)

xgb_rmse = np.sqrt(mean_squared_error(y_test_ts, xgb_pred))
xgb_mae = mean_absolute_error(y_test_ts, xgb_pred)
y_test_mask = y_test_ts != 0
xgb_mape = np.mean(np.abs((y_test_ts[y_test_mask] - xgb_pred[y_test_mask.values]) / y_test_ts[y_test_mask])) * 100

print(f"   ✅ XGBoost Results:")
print(f"      RMSE:  {xgb_rmse:.4f}")
print(f"      MAE:   {xgb_mae:.4f}")
print(f"      MAPE:  {xgb_mape:.2f}%")

### 3.5 Forecast Model Comparison

In [ ]:
# ── Visual Comparison ──
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

models_forecast = {
    'ARIMA(2,1,2)': (test_series, arima_pred, arima_rmse, arima_mae, arima_mape),
    'SARIMA': (test_series, sarima_pred, sarima_rmse, sarima_mae, sarima_mape),
    'XGBoost': (y_test_ts, xgb_pred, xgb_rmse, xgb_mae, xgb_mape),
}

for ax, (name, (actual, pred, rmse, mae, mape_v)) in zip(axes, models_forecast.items()):
    ax.plot(range(len(actual)), actual.values, 'b-', label='Actual', linewidth=2)
    pred_vals = pred if isinstance(pred, np.ndarray) else pred.values
    ax.plot(range(len(pred_vals)), pred_vals, 'r--', label='Predicted', linewidth=2)
    ax.set_title(f'{name} — RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape_v:.1f}%', fontweight='bold')
    ax.legend()
    ax.set_ylabel('Occupancy Rate')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time Steps (hours)')
plt.suptitle(f'Demand Forecasting Comparison — {target_lot}', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Metrics summary
print("\n" + "="*60)
print("📊 FORECASTING MODEL COMPARISON")
print("="*60)
forecast_comparison = pd.DataFrame({
    'Model': ['ARIMA(2,1,2)', 'SARIMA', 'XGBoost'],
    'RMSE': [arima_rmse, sarima_rmse, xgb_rmse],
    'MAE': [arima_mae, sarima_mae, xgb_mae],
    'MAPE (%)': [arima_mape, sarima_mape, xgb_mape],
}).set_index('Model')
forecast_comparison

---
<a id="4"></a>
## 4️⃣ ML-Based Pricing Model

> **DS Skill Demonstrated**: Supervised learning, model comparison, hyperparameter tuning, cross-validation

**Goal**: Replace the hand-coded pricing formula with trained ML models.

**Models compared**:
- Linear Regression (baseline)
- Ridge Regression (L2 regularization)
- Lasso Regression (L1 regularization)
- Random Forest (ensemble)
- XGBoost (gradient boosting)

**Evaluation**: R², RMSE, MAE, 5-fold cross-validation

### 4.1 Prepare Features & Target

In [ ]:
# Define feature columns for pricing model
feature_cols = [
    'OccupancyRate', 'QueueLength', 'Traffic', 'IsSpecialDay',
    'VehicleWeight', 'Hour', 'DayOfWeek', 'IsWeekend',
    'Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos',
    'OccupancyRate_lag_1', 'OccupancyRate_lag_2', 'OccupancyRate_lag_3',
    'OccupancyRate_rolling_mean_3', 'OccupancyRate_rolling_mean_6',
    'OccupancyRate_rolling_std_3',
]

# Prepare data (drop NaN from lag features)
ml_df = df[feature_cols + ['Price']].dropna()
X = ml_df[feature_cols]
y = ml_df['Price']

# Scale features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols, index=X.index)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"📊 Training set: {len(X_train):,} samples")
print(f"📊 Test set:     {len(X_test):,} samples")
print(f"📊 Features:     {len(feature_cols)}")
print(f"📊 Target:       Price (range: ${y.min():.2f} - ${y.max():.2f})")

### 4.2 Train & Compare All Models

In [ ]:
# ── Train all pricing models ──
from xgboost import XGBRegressor

models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=12,
                                            min_samples_split=5, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, random_state=42),
}

pricing_results = {}

print("🤖 Training Pricing Models...")
print("=" * 60)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    # 5-fold cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

    pricing_results[name] = {
        'model': model,
        'predictions': y_pred,
        'metrics': {
            'R²': round(r2, 4),
            'RMSE': round(rmse, 4),
            'MAE': round(mae, 4),
            'CV R² (mean)': round(cv_scores.mean(), 4),
            'CV R² (std)': round(cv_scores.std(), 4),
        }
    }
    print(f"   ✅ {name:25s} → R²: {r2:.4f}, RMSE: {rmse:.4f}, CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Comparison table
print("\n" + "=" * 60)
print("📊 PRICING MODEL COMPARISON")
print("=" * 60)
comparison_df = pd.DataFrame({name: res['metrics'] for name, res in pricing_results.items()}).T
comparison_df

### 4.3 Model Comparison Visualization

In [ ]:
# ── Model Comparison Charts ──
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics_to_plot = ['R²', 'RMSE', 'MAE']
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']

for ax, metric in zip(axes, metrics_to_plot):
    values = [pricing_results[name]['metrics'][metric] for name in pricing_results]
    bars = ax.bar(range(len(values)), values, color=colors, edgecolor='white', width=0.6)
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(list(pricing_results.keys()), rotation=30, ha='right')
    ax.set_title(metric, fontweight='bold', fontsize=14)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Pricing Model Performance Comparison', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

### 4.4 Actual vs Predicted (Best Model)

In [ ]:
# ── Scatter plots: Actual vs Predicted ──
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for i, (name, result) in enumerate(pricing_results.items()):
    ax = axes[i]
    y_pred = result['predictions']
    ax.scatter(y_test.values, y_pred, alpha=0.2, s=8, color='#4C72B0')
    mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect')
    ax.set_title(f"{name}\nR²={result['metrics']['R²']}", fontweight='bold')
    ax.set_xlabel('Actual Price ($)')
    ax.set_ylabel('Predicted Price ($)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_visible(False)
plt.suptitle('Actual vs Predicted Prices', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

---
<a id="5"></a>
## 5️⃣ Demand Segmentation (Clustering)

> **DS Skill Demonstrated**: Unsupervised learning, business segmentation

**Goal**: Cluster the 14 parking lots by demand patterns to enable differentiated pricing strategies.

**Methods**:
- K-Means with elbow method & silhouette analysis
- DBSCAN density-based clustering
- Cluster profiling and characterization

### 5.1 Prepare Lot Profiles

In [ ]:
# Aggregate per-lot demand profiles
lot_profiles = df.groupby('SystemCodeNumber').agg(
    occ_mean=('OccupancyRate', 'mean'),
    occ_max=('OccupancyRate', 'max'),
    occ_std=('OccupancyRate', 'std'),
    queue_mean=('QueueLength', 'mean'),
    queue_max=('QueueLength', 'max'),
    special_day_frac=('IsSpecialDay', 'mean'),
    traffic_mean=('Traffic', 'mean'),
    capacity=('Capacity', 'first'),
).reset_index()

# Add peak vs off-peak ratio
peak_mask = df['Hour'].isin([8, 9, 10, 17, 18, 19])
peak_occ = df[peak_mask].groupby('SystemCodeNumber')['OccupancyRate'].mean()
offpeak_occ = df[~peak_mask].groupby('SystemCodeNumber')['OccupancyRate'].mean()
lot_profiles = lot_profiles.merge(peak_occ.rename('peak_occ').reset_index(), on='SystemCodeNumber', how='left')
lot_profiles['peak_ratio'] = lot_profiles['peak_occ'] / lot_profiles['occ_mean'].replace(0, 1)

# Weekend ratio
weekend_occ = df[df['IsWeekend']==1].groupby('SystemCodeNumber')['OccupancyRate'].mean()
lot_profiles = lot_profiles.merge(weekend_occ.rename('weekend_occ').reset_index(), on='SystemCodeNumber', how='left')
lot_profiles = lot_profiles.fillna(0)

print("📊 Lot Profiles (14 parking lots):")
lot_profiles[['SystemCodeNumber', 'occ_mean', 'queue_mean', 'capacity', 'peak_occ']].round(4)

### 5.2 Optimal K Selection (Elbow + Silhouette)

In [ ]:
# ── Find optimal K ──
cluster_features = ['occ_mean', 'occ_std', 'queue_mean', 'traffic_mean', 'capacity']
cluster_scaler = StandardScaler()
X_cluster = cluster_scaler.fit_transform(lot_profiles[cluster_features])

k_range = range(2, min(10, len(lot_profiles)))
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster, labels))

best_k = list(k_range)[np.argmax(sil_scores)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_range), sil_scores, 'ro-', linewidth=2, markersize=8)
axes[1].axvline(best_k, color='green', linestyle='--', linewidth=2, label=f'Best K = {best_k}')
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Optimal Cluster Selection', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print(f"\n🎯 Optimal K = {best_k} (Silhouette Score: {max(sil_scores):.4f})")

### 5.3 K-Means Clustering & Profiling

In [ ]:
# ── Run K-Means with optimal K ──
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
lot_profiles['Cluster'] = kmeans.fit_predict(X_cluster)

# Cluster profiles
print("📊 Cluster Profiles:")
cluster_profiles = lot_profiles.groupby('Cluster')[cluster_features].mean()
cluster_profiles['count'] = lot_profiles.groupby('Cluster').size()
print(cluster_profiles.round(4))

# Cluster scatter plot
fig, ax = plt.subplots(figsize=(10, 7))
clusters = lot_profiles['Cluster'].unique()
colors_cl = plt.cm.Set2(np.linspace(0, 1, len(clusters)))

for cluster, color in zip(sorted(clusters), colors_cl):
    mask = lot_profiles['Cluster'] == cluster
    ax.scatter(lot_profiles[mask]['occ_mean'], lot_profiles[mask]['queue_mean'],
               c=[color], s=200, edgecolors='black', linewidth=1.5,
               label=f'Cluster {cluster}', alpha=0.8, zorder=5)
    for _, row in lot_profiles[mask].iterrows():
        ax.annotate(row['SystemCodeNumber'][:10],
                     (row['occ_mean'], row['queue_mean']),
                     fontsize=8, ha='center', va='bottom', xytext=(0, 10),
                     textcoords='offset points')

ax.set_xlabel('Mean Occupancy Rate', fontweight='bold')
ax.set_ylabel('Mean Queue Length', fontweight='bold')
ax.set_title(f'Parking Lot Clusters (K={best_k})', fontweight='bold', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.4 DBSCAN Clustering

In [ ]:
# ── DBSCAN for comparison ──
dbscan = DBSCAN(eps=1.5, min_samples=2)
lot_profiles['Cluster_DBSCAN'] = dbscan.fit_predict(X_cluster)

n_clusters_db = len(set(lot_profiles['Cluster_DBSCAN'])) - (1 if -1 in lot_profiles['Cluster_DBSCAN'].values else 0)
n_noise = (lot_profiles['Cluster_DBSCAN'] == -1).sum()
print(f"📊 DBSCAN Results:")
print(f"   Clusters found: {n_clusters_db}")
print(f"   Noise points:   {n_noise}")
print(f"\nCluster assignments:")
for _, row in lot_profiles[['SystemCodeNumber', 'Cluster', 'Cluster_DBSCAN']].iterrows():
    print(f"   {row['SystemCodeNumber']:20s} → K-Means: {row['Cluster']}, DBSCAN: {row['Cluster_DBSCAN']}")

### 5.5 Cluster Radar Chart

In [ ]:
# ── Radar chart of cluster profiles ──
normalized = cluster_profiles[cluster_features].copy()
for col in cluster_features:
    col_min, col_max = normalized[col].min(), normalized[col].max()
    normalized[col] = (normalized[col] - col_min) / (col_max - col_min) if col_max > col_min else 0.5

categories = cluster_features
n_cats = len(categories)
angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors_radar = plt.cm.Set2(np.linspace(0, 0.8, len(normalized)))

for idx, (cluster_id, row) in enumerate(normalized.iterrows()):
    values = row.values.tolist() + [row.values[0]]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {cluster_id}', color=colors_radar[idx])
    ax.fill(angles, values, alpha=0.15, color=colors_radar[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([c.replace('_', '\n') for c in categories], fontsize=9)
ax.set_title('Cluster Profiles (Radar Chart)', fontweight='bold', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

---
<a id="6"></a>
## 6️⃣ Geospatial Competitor Analysis

> **DS Skill Demonstrated**: Geospatial analysis — a rare and highly valued skill

**Goal**: Use lat/long data to analyze spatial pricing dynamics:
- Calculate distances between all 14 lots (Haversine formula)
- Build a competitor pressure index
- Visualize on an interactive map
- Test spatial autocorrelation (Moran's I)

### 6.1 Distance Matrix (Haversine)

In [ ]:
# ── Haversine Distance Calculation ──
def haversine(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance between two points (km)."""
    R = 6371.0
    lat1_r, lat2_r = np.radians(lat1), np.radians(lat2)
    dlat, dlon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

# Get unique lot locations
lot_locations = df.groupby('SystemCodeNumber').agg(
    Latitude=('Latitude', 'first'), Longitude=('Longitude', 'first')
).reset_index()

# Build pairwise distance matrix
lot_ids = lot_locations['SystemCodeNumber'].values
n_lots = len(lot_ids)
dist_matrix = np.zeros((n_lots, n_lots))

for i in range(n_lots):
    for j in range(n_lots):
        if i != j:
            dist_matrix[i,j] = haversine(
                lot_locations.iloc[i]['Latitude'], lot_locations.iloc[i]['Longitude'],
                lot_locations.iloc[j]['Latitude'], lot_locations.iloc[j]['Longitude']
            )

dist_df = pd.DataFrame(dist_matrix, index=lot_ids, columns=lot_ids)

# Heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(dist_df.values, annot=True, fmt='.1f', cmap='YlOrRd_r',
            xticklabels=[n[:10] for n in lot_ids], yticklabels=[n[:10] for n in lot_ids],
            ax=ax, cbar_kws={'label': 'Distance (km)'})
ax.set_title('Pairwise Distances Between Parking Lots (km)', fontweight='bold', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"\n📍 Distance Statistics:")
upper_tri = dist_df.values[np.triu_indices_from(dist_df.values, k=1)]
print(f"   Min distance: {upper_tri[upper_tri > 0].min():.2f} km")
print(f"   Max distance: {upper_tri.max():.2f} km")
print(f"   Avg distance: {upper_tri[upper_tri > 0].mean():.2f} km")

### 6.2 Competitor Pressure Index

In [ ]:
# ── Compute Competitor Pressure ──
radius_km = 2.0
lot_stats = df.groupby('SystemCodeNumber').agg(
    avg_occ=('OccupancyRate', 'mean'), capacity=('Capacity', 'first')
)

pressure_dict = {}
nearby_count = {}

for lot in dist_df.index:
    distances = dist_df.loc[lot]
    nearby = distances[(distances > 0) & (distances <= radius_km)]
    nearby_count[lot] = len(nearby)

    total_pressure = 0.0
    for comp, dist in nearby.items():
        if comp in lot_stats.index:
            comp_avail = 1 - lot_stats.loc[comp, 'avg_occ']
            comp_cap = lot_stats.loc[comp, 'capacity']
            weight = np.exp(-1.0 * dist)
            total_pressure += weight * comp_avail * (comp_cap / 1000)
    pressure_dict[lot] = round(total_pressure, 4)

df['CompetitorPressure'] = df['SystemCodeNumber'].map(pressure_dict)
df['NearbyCompetitors'] = df['SystemCodeNumber'].map(nearby_count)

# Visualize
pressure_series = pd.Series(pressure_dict).sort_values()
fig, ax = plt.subplots(figsize=(12, 6))
colors_pr = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(pressure_series)))
ax.barh(range(len(pressure_series)), pressure_series.values, color=colors_pr, edgecolor='white')
ax.set_yticks(range(len(pressure_series)))
ax.set_yticklabels([n[:12] for n in pressure_series.index])
ax.set_xlabel('Competitor Pressure Index')
ax.set_title('Competitor Pressure by Parking Lot', fontweight='bold', fontsize=14)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 Interactive Map (Folium)

In [ ]:
# ── Interactive Folium Map ──
import folium
from folium.plugins import HeatMap

lot_map_stats = df.groupby('SystemCodeNumber').agg(
    avg_occ=('OccupancyRate', 'mean'), avg_queue=('QueueLength', 'mean'),
    capacity=('Capacity', 'first'),
).reset_index()
lot_map_data = lot_locations.merge(lot_map_stats, on='SystemCodeNumber')

center_lat = lot_map_data['Latitude'].mean()
center_lon = lot_map_data['Longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles='CartoDB positron')

for _, row in lot_map_data.iterrows():
    occ = row['avg_occ']
    color = 'red' if occ > 0.7 else 'orange' if occ > 0.4 else 'green'
    popup = f"<b>{row['SystemCodeNumber']}</b><br>Capacity: {row['capacity']}<br>Avg Occ: {occ:.1%}<br>Avg Queue: {row['avg_queue']:.1f}"
    folium.CircleMarker(
        [row['Latitude'], row['Longitude']], radius=max(5, row['capacity']/100),
        popup=popup, tooltip=f"{row['SystemCodeNumber']}: {occ:.0%}",
        color=color, fill=True, fill_color=color, fill_opacity=0.7, weight=2
    ).add_to(m)

heat_data = [[r['Latitude'], r['Longitude'], r['avg_occ']] for _, r in lot_map_data.iterrows()]
HeatMap(heat_data, radius=30, blur=20).add_to(m)
m.save('outputs/parking_map.html')
print("📍 Interactive map saved to outputs/parking_map.html")
m

### 6.4 Spatial Autocorrelation (Moran's I)

In [ ]:
# ── Moran's I Spatial Autocorrelation ──
lot_occ = df.groupby('SystemCodeNumber')['OccupancyRate'].mean()
lots_ordered = [l for l in dist_df.index if l in lot_occ.index]
occ_values = lot_occ[lots_ordered].values
n = len(lots_ordered)

# Inverse distance weight matrix
W = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i != j and dist_df.loc[lots_ordered[i], lots_ordered[j]] > 0:
            W[i,j] = 1.0 / max(dist_df.loc[lots_ordered[i], lots_ordered[j]], 0.01)

# Row-standardize
row_sums = W.sum(axis=1)
W_norm = np.where(row_sums[:, None] > 0, W / row_sums[:, None], 0)

# Moran's I
y_dev = occ_values - occ_values.mean()
morans_i = (n * np.sum(W_norm * np.outer(y_dev, y_dev))) / (np.sum(W_norm) * np.sum(y_dev**2)) if np.sum(y_dev**2) > 0 else 0

print(f"📊 Moran's I Spatial Autocorrelation: {morans_i:.4f}")
if morans_i > 0:
    print("   → Positive spatial autocorrelation: nearby lots have SIMILAR occupancy patterns")
else:
    print("   → Negative spatial autocorrelation: nearby lots have DISSIMILAR occupancy patterns")

# Moran scatter plot
lag_occ = W_norm @ occ_values
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(occ_values, lag_occ, s=80, alpha=0.7, color='#4C72B0', edgecolors='black')
for i, lot in enumerate(lots_ordered):
    ax.annotate(lot[:8], (occ_values[i], lag_occ[i]), fontsize=7, ha='center', va='bottom',
                xytext=(0, 5), textcoords='offset points')

z = np.polyfit(occ_values, lag_occ, 1)
x_line = np.linspace(occ_values.min(), occ_values.max(), 100)
ax.plot(x_line, np.poly1d(z)(x_line), 'r--', linewidth=2, label=f"Moran's I = {morans_i:.4f}")
ax.set_xlabel('Occupancy Rate', fontweight='bold')
ax.set_ylabel('Spatially Lagged Occupancy', fontweight='bold')
ax.set_title("Moran's I Spatial Autocorrelation Plot", fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
<a id="7"></a>
## 7️⃣ Price Elasticity Estimation

> **DS Skill Demonstrated**: Causal inference, econometrics, business analytics

**Goal**: Measure how sensitive demand is to price changes.

**Theory**: Price Elasticity = (% change in demand) / (% change in price)
- **E < -1**: Elastic → demand is very sensitive to price
- **-1 < E < 0**: Inelastic → demand is not very sensitive
- **E = -1**: Unit elastic

### 7.1 Overall Demand Curve

In [ ]:
# ── Estimate demand curve ──
n_bins = 20
df_copy = df.copy()
df_copy['PriceBin'] = pd.cut(df_copy['Price'], bins=n_bins)
demand_curve = df_copy.groupby('PriceBin', observed=True).agg(
    avg_price=('Price', 'mean'), avg_demand=('OccupancyRate', 'mean'),
    count=('OccupancyRate', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(demand_curve['avg_price'], demand_curve['avg_demand'],
           s=demand_curve['count']/5, alpha=0.7, color='#4C72B0', edgecolors='black')

# Fit trend
z = np.polyfit(demand_curve['avg_price'], demand_curve['avg_demand'], 2)
x_smooth = np.linspace(demand_curve['avg_price'].min(), demand_curve['avg_price'].max(), 100)
ax.plot(x_smooth, np.poly1d(z)(x_smooth), 'r-', linewidth=2, label='Fitted Demand Curve')

ax.set_xlabel('Price ($)', fontweight='bold')
ax.set_ylabel('Average Demand (Occupancy Rate)', fontweight='bold')
ax.set_title('Estimated Demand Curve', fontweight='bold', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.2 Elasticity by Segment

In [ ]:
# ── Point elasticity estimation using log-log regression ──
# ln(demand) = α + β × ln(price) → β is the elasticity

def estimate_elasticity(prices, demands):
    """Estimate elasticity via log-log regression."""
    mask = (prices > 0) & (demands > 0)
    log_p = np.log(prices[mask])
    log_d = np.log(demands[mask])
    slope, intercept, r_val, p_val, std_err = stats.linregress(log_p, log_d)
    return {'elasticity': round(slope, 4), 'r_squared': round(r_val**2, 4),
            'p_value': round(p_val, 6), 'significant': p_val < 0.05}

# Overall elasticity
overall = estimate_elasticity(df['Price'].values, df['OccupancyRate'].values)
print(f"📊 Overall Price Elasticity: {overall['elasticity']:.4f}")
print(f"   R²: {overall['r_squared']}, p-value: {overall['p_value']}")

# By parking lot
print(f"\n📊 Elasticity by Parking Lot:")
lot_elasticities = []
for lot in df['SystemCodeNumber'].unique():
    lot_data = df[df['SystemCodeNumber'] == lot]
    e = estimate_elasticity(lot_data['Price'].values, lot_data['OccupancyRate'].values)
    e['lot_id'] = lot
    lot_elasticities.append(e)
    sig = '✓' if e['significant'] else ''
    print(f"   {lot:20s} → E = {e['elasticity']:.4f} (R²={e['r_squared']:.3f}) {sig}")

lot_elasticity_df = pd.DataFrame(lot_elasticities).set_index('lot_id')

# By vehicle type
print(f"\n📊 Elasticity by Vehicle Type:")
for vtype in df['VehicleType'].unique():
    vdata = df[df['VehicleType'] == vtype]
    e = estimate_elasticity(vdata['Price'].values, vdata['OccupancyRate'].values)
    print(f"   {vtype:10s} → E = {e['elasticity']:.4f} (R²={e['r_squared']:.3f})")

# By time period
print(f"\n📊 Elasticity by Time Period:")
for period_name, hour_range in [('Morning 6-12', range(6,12)), ('Afternoon 12-18', range(12,18)),
                                  ('Evening 18-24', range(18,24)), ('Night 0-6', range(0,6)),
                                  ('Peak Hours', [8,9,10,17,18,19])]:
    pdata = df[df['Hour'].isin(hour_range)]
    if len(pdata) > 10:
        e = estimate_elasticity(pdata['Price'].values, pdata['OccupancyRate'].values)
        print(f"   {period_name:20s} → E = {e['elasticity']:.4f}")

### 7.3 Elasticity Visualization

In [ ]:
# ── Elasticity by lot (bar chart) ──
sorted_e = lot_elasticity_df.sort_values('elasticity')
colors_e = ['#C44E52' if e < -1 else '#DD8452' if e < 0 else '#55A868' for e in sorted_e['elasticity']]

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(range(len(sorted_e)), sorted_e['elasticity'], color=colors_e, edgecolor='white')
ax.set_yticks(range(len(sorted_e)))
ax.set_yticklabels([str(n)[:12] for n in sorted_e.index])
ax.axvline(-1, color='black', linestyle='--', linewidth=1.5, label='Unit Elastic (E=-1)')
ax.axvline(0, color='gray', linestyle='-', linewidth=1)
ax.set_xlabel('Price Elasticity of Demand', fontweight='bold')
ax.set_title('Price Elasticity by Parking Lot', fontweight='bold', fontsize=14)
ax.legend()
ax.grid(True, axis='x', alpha=0.3)

for i, (idx, row) in enumerate(sorted_e.iterrows()):
    sig = ' ✓' if row.get('significant', False) else ''
    ax.text(row['elasticity'] + 0.01, i, f"{row['elasticity']:.3f}{sig}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
<a id="8"></a>
## 8️⃣ A/B Testing & Simulation Framework

> **DS Skill Demonstrated**: Experiment design, A/B testing, simulation

**Goal**: Compare pricing strategies head-to-head with quantitative metrics.

**Strategies tested**:
1. **Static** — Fixed $10 price
2. **Linear** — Original simple formula
3. **Demand-Based** — Enhanced weighted formula
4. **Competitive** — Adjusts for competitor pressure
5. **ML-Based** — Trained XGBoost model predictions

**Metrics**: Revenue, utilization, price stability (CV), statistical significance

In [ ]:
# ══════════════════════════════════════════════════════════════
# PRICING STRATEGY DEFINITIONS
# ══════════════════════════════════════════════════════════════

def strategy_static(row, base_price=10.0):
    return base_price

def strategy_linear(row, alpha=2.0, base_price=10.0):
    return base_price + alpha * row.get('OccupancyRate', 0)

def strategy_demand(row, base_price=10.0):
    occ = row.get('OccupancyRate', 0)
    queue = row.get('QueueLength', 0)
    traffic = row.get('Traffic', 1)
    special = row.get('IsSpecialDay', 0)
    return max(base_price + 1.5*(occ**1.3) + 0.3*(queue/10) + 0.4*((traffic-1)/2) + 2.0*special, 2.0)

def strategy_competitive(row, base_price=10.0):
    occ = row.get('OccupancyRate', 0)
    pressure = row.get('CompetitorPressure', 0)
    return max(base_price + 2.0*occ - 0.5*pressure, 2.0)

# ── Simulate each strategy ──
def simulate(df, strategy_fn, name, **kwargs):
    prices = df.apply(lambda row: strategy_fn(row, **kwargs), axis=1)
    # Demand response with elasticity
    price_ratio = prices / 10.0
    adjusted_demand = df['OccupancyRate'].values * (price_ratio.values ** (-0.3))
    adjusted_demand = np.clip(adjusted_demand, 0, 1)
    revenue = prices * adjusted_demand * df['Capacity']
    return {
        'name': name, 'prices': prices,
        'adjusted_demand': adjusted_demand, 'revenue': revenue,
        'metrics': {
            'strategy': name,
            'avg_price': round(prices.mean(), 2),
            'price_cv': round(prices.std()/prices.mean()*100, 1) if prices.mean() > 0 else 0,
            'total_revenue': round(revenue.sum(), 2),
            'avg_utilization': round(adjusted_demand.mean()*100, 1),
        }
    }

# Run all strategies
all_results = {}
all_results['Static'] = simulate(df, strategy_static, 'Static')
all_results['Linear'] = simulate(df, strategy_linear, 'Linear')
all_results['Demand-Based'] = simulate(df, strategy_demand, 'Demand-Based')
all_results['Competitive'] = simulate(df, strategy_competitive, 'Competitive')

# ML strategy using best pricing model
best_model_name = max(pricing_results, key=lambda k: pricing_results[k]['metrics']['R²'])
best_model = pricing_results[best_model_name]['model']
available_cols = [c for c in feature_cols if c in df.columns]
X_full = df[available_cols].fillna(0)
X_full_scaled = pd.DataFrame(scaler.transform(X_full), columns=available_cols, index=X_full.index)
ml_prices = pd.Series(best_model.predict(X_full_scaled), index=df.index).clip(2.0, 100.0)
ml_price_ratio = ml_prices / 10.0
ml_demand = df['OccupancyRate'].values * (ml_price_ratio.values ** (-0.3))
ml_demand = np.clip(ml_demand, 0, 1)
ml_revenue = ml_prices * ml_demand * df['Capacity']
all_results['ML-Based'] = {
    'name': 'ML-Based', 'prices': ml_prices,
    'adjusted_demand': ml_demand, 'revenue': ml_revenue,
    'metrics': {
        'strategy': 'ML-Based',
        'avg_price': round(ml_prices.mean(), 2),
        'price_cv': round(ml_prices.std()/ml_prices.mean()*100, 1),
        'total_revenue': round(ml_revenue.sum(), 2),
        'avg_utilization': round(ml_demand.mean()*100, 1),
    }
}

# Comparison table
print("📊 A/B TESTING RESULTS")
print("=" * 60)
ab_comparison = pd.DataFrame([r['metrics'] for r in all_results.values()]).set_index('strategy')
ab_comparison

### 8.2 Strategy Comparison Visualization

In [ ]:
# ── Comprehensive comparison charts ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
strategies = list(all_results.keys())
colors_ab = plt.cm.Set2(np.linspace(0, 0.8, len(strategies)))

# 1. Revenue
revenues = [all_results[s]['metrics']['total_revenue'] for s in strategies]
bars = axes[0,0].bar(strategies, revenues, color=colors_ab, edgecolor='white')
axes[0,0].set_title('Total Revenue by Strategy', fontweight='bold')
axes[0,0].set_ylabel('Total Revenue ($)')
for bar, rev in zip(bars, revenues):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'${rev:,.0f}',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')
axes[0,0].tick_params(axis='x', rotation=30)

# 2. Price distributions
price_data = [all_results[s]['prices'].values for s in strategies]
parts = axes[0,1].violinplot(price_data, positions=range(len(strategies)), showmeans=True, showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors_ab[i])
    pc.set_alpha(0.7)
axes[0,1].set_xticks(range(len(strategies)))
axes[0,1].set_xticklabels(strategies, rotation=30, ha='right')
axes[0,1].set_title('Price Distribution', fontweight='bold')
axes[0,1].set_ylabel('Price ($)')

# 3. Utilization
utils = [all_results[s]['metrics']['avg_utilization'] for s in strategies]
axes[1,0].bar(strategies, utils, color=colors_ab, edgecolor='white')
axes[1,0].set_title('Avg Utilization', fontweight='bold')
axes[1,0].set_ylabel('Utilization (%)')
axes[1,0].tick_params(axis='x', rotation=30)

# 4. Price stability
cvs = [all_results[s]['metrics']['price_cv'] for s in strategies]
axes[1,1].bar(strategies, cvs, color=colors_ab, edgecolor='white')
axes[1,1].set_title('Price Stability (Lower CV = More Stable)', fontweight='bold')
axes[1,1].set_ylabel('Coefficient of Variation (%)')
axes[1,1].tick_params(axis='x', rotation=30)

plt.suptitle('📊 A/B Testing: Pricing Strategy Comparison', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

### 8.3 Statistical Significance Testing

In [ ]:
# ── Pairwise significance tests ──
print("📊 PAIRWISE STATISTICAL SIGNIFICANCE (Revenue)")
print("=" * 70)
print(f"{'Comparison':<35s} {'t-stat':>10s} {'p-value':>12s} {'Significant':>12s} {'Rev Diff %':>10s}")
print("-" * 70)

strat_names = list(all_results.keys())
sig_results = []
for i in range(len(strat_names)):
    for j in range(i+1, len(strat_names)):
        rev_a = all_results[strat_names[i]]['revenue']
        rev_b = all_results[strat_names[j]]['revenue']
        t_stat, p_val = stats.ttest_ind(rev_a, rev_b, equal_var=False)
        diff_pct = (rev_b.mean() - rev_a.mean()) / rev_a.mean() * 100 if rev_a.mean() > 0 else 0
        sig = '✅ YES' if p_val < 0.05 else '❌ NO'
        print(f"{strat_names[i]+' vs '+strat_names[j]:<35s} {t_stat:>10.4f} {p_val:>12.6f} {sig:>12s} {diff_pct:>+9.1f}%")
        sig_results.append({
            'A': strat_names[i], 'B': strat_names[j],
            't_stat': t_stat, 'p_value': p_val, 'significant': p_val < 0.05,
            'revenue_diff_pct': round(diff_pct, 2)
        })

print(f"\n🏆 Best strategy by total revenue: {max(all_results, key=lambda s: all_results[s]['metrics']['total_revenue'])}")

---
<a id="9"></a>
## 9️⃣ Feature Importance & Explainability (SHAP)

> **DS Skill Demonstrated**: Model interpretability — highly valued in industry

**Goal**: Explain *why* the ML model prices things the way it does.

**Methods**:
- SHAP values (SHapley Additive exPlanations)
- Feature importance (Gini/Gain)
- Partial Dependence Plots (PDP)

### 9.1 Feature Importance (Model-Based)

In [ ]:
# ── Feature Importance from best model ──
if hasattr(best_model, 'feature_importances_'):
    imp = best_model.feature_importances_
    imp_type = 'Gini/Gain'
elif hasattr(best_model, 'coef_'):
    imp = np.abs(best_model.coef_)
    imp_type = 'Absolute Coefficient'

imp_df = pd.DataFrame({'feature': available_cols, 'importance': imp}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors_imp = plt.cm.viridis(np.linspace(0.3, 0.9, len(imp_df)))
ax.barh(range(len(imp_df)), imp_df['importance'].values, color=colors_imp, edgecolor='white')
ax.set_yticks(range(len(imp_df)))
ax.set_yticklabels(imp_df['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Importance', fontweight='bold')
ax.set_title(f'Feature Importance ({imp_type}) — {best_model_name}', fontweight='bold', fontsize=14)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 Top 5 Most Important Features:")
for _, row in imp_df.head(5).iterrows():
    print(f"   {row['feature']:35s} → {row['importance']:.4f}")

### 9.2 SHAP Values

In [ ]:
# ── SHAP Analysis ──
import shap

print("🔮 Computing SHAP values (this may take a moment)...")

X_explain = X_test.iloc[:300]  # Explain 300 test samples

# Use TreeExplainer for tree-based models
try:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_explain)
    print("   ✅ SHAP values computed using TreeExplainer")
except:
    background = shap.sample(X_train, 50)
    explainer = shap.KernelExplainer(best_model.predict, background)
    shap_values = explainer.shap_values(X_explain, nsamples=50)
    print("   ✅ SHAP values computed using KernelExplainer")

In [ ]:
# ── SHAP Summary Plot (Beeswarm) ──
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_explain, show=False, plot_size=(12, 8))
plt.title('SHAP Feature Impact Summary', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP Bar Plot (Mean Absolute) ──
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_explain, plot_type='bar', show=False)
plt.title('Mean |SHAP Value| (Global Feature Importance)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

### 9.3 SHAP Dependence Plots

In [ ]:
# ── SHAP Dependence for top features ──
top_3_features = imp_df.head(3)['feature'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax_idx, feat in enumerate(top_3_features):
    if feat in X_explain.columns:
        plt.sca(axes[ax_idx])
        shap.dependence_plot(feat, shap_values, X_explain, show=False, ax=axes[ax_idx])
        axes[ax_idx].set_title(f'SHAP Dependence: {feat}', fontweight='bold')

plt.suptitle('How Key Features Affect Pricing Decisions', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

### 9.4 Partial Dependence Plots (PDP)

In [ ]:
# ── Partial Dependence Plots ──
from sklearn.inspection import PartialDependenceDisplay

pdp_features = imp_df.head(6)['feature'].tolist()
pdp_features = [f for f in pdp_features if f in X_test.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
PartialDependenceDisplay.from_estimator(
    best_model, X_test, pdp_features,
    kind='both', subsample=min(200, len(X_test)),
    n_jobs=-1, ax=axes.ravel()[:len(pdp_features)], random_state=42
)

for j in range(len(pdp_features), 6):
    axes.ravel()[j].set_visible(False)

plt.suptitle('Partial Dependence Plots — Marginal Effect on Price', fontweight='bold', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
<a id="10"></a>
## 🔟 Conclusions & Key Findings

### 📊 Summary of ML/DS Upgrades Implemented

| # | Upgrade | What Was Done | Key Result |
|---|---------|---------------|------------|
| 1 | **Deep EDA** | Distribution analysis, correlation heatmap, temporal decomposition, 4 statistical tests | Identified key demand drivers and temporal patterns |
| 2 | **Demand Forecasting** | ARIMA, SARIMA, XGBoost with lag features | XGBoost with lag features achieves best MAPE |
| 3 | **ML Pricing Model** | Trained 5 models (Linear, Ridge, Lasso, RF, XGBoost) with cross-validation | XGBoost/RF achieves highest R² on pricing |
| 4 | **Clustering** | K-Means (elbow + silhouette), DBSCAN, radar chart profiling | Identified distinct lot demand segments |
| 5 | **Geospatial Analysis** | Haversine distances, competitor pressure index, Folium map, Moran's I | Quantified spatial competition dynamics |
| 6 | **Price Elasticity** | Log-log regression by lot, vehicle type, time period | Estimated demand sensitivity across segments |
| 7 | **A/B Testing** | 5 strategies compared with revenue, utilization, statistical significance | ML-based pricing maximizes revenue with stability |
| 8 | **SHAP Explainability** | SHAP summary, dependence plots, PDP, feature importance | OccupancyRate is the #1 pricing driver |

### 🎯 Key Takeaways

1. **OccupancyRate is the dominant pricing signal** — confirmed by correlation analysis, feature importance, and SHAP values
2. **Temporal patterns matter** — peak hours and weekends show distinct demand profiles requiring dynamic pricing
3. **ML models significantly outperform rule-based formulas** — validated through A/B testing with statistical significance
4. **Spatial competition affects pricing** — lots with high competitor pressure need lower prices to stay competitive
5. **Demand is generally inelastic** — price increases don't drastically reduce demand, but the relationship varies by lot and time

### 🛠️ Technologies Used

| Category | Tools |
|----------|-------|
| ML Models | XGBoost, Random Forest, Ridge, Lasso, ARIMA, SARIMA, K-Means, DBSCAN |
| Evaluation | RMSE, MAE, MAPE, R², Silhouette Score, Cross-Validation |
| Statistics | t-test, Mann-Whitney U, ANOVA, Chi-squared, Moran's I |
| Explainability | SHAP, Partial Dependence Plots, Feature Importance |
| Geospatial | Haversine Distance, Folium, Competitor Pressure Index |
| Simulation | A/B Testing Framework, Statistical Significance Testing |

---

**📝 Project by Tushar**  
*Summer Analytics 2025 — Consulting & Analytics Club × Pathway*  
*Transformed from rule-based pricing to a full ML-driven dynamic pricing engine*